In [ ]:
# =============================================================================
# TASK 3 — HYPERPARAMETER TUNING (RandomizedSearchCV)
# Operational Risk Level Classification (Low / Moderate / High)
# =============================================================================


from sklearn.model_selection import RandomizedSearchCV
import time

RANDOM_STATE = 42

# =============================================================================
# HYPERPARAMETER SEARCH SPACES
# =============================================================================

param_grids = {

    # ── Logistic Regression ──────────────────────────────────────────────────
    "Logistic Regression": {
        "clf__C"        : [0.001, 0.01, 0.1, 0.5, 1.0, 5.0, 10.0],
        "clf__solver"   : ["lbfgs"],
        "clf__max_iter" : [500, 1000, 2000],
    },

    # ── Decision Tree ────────────────────────────────────────────────────────
    "Decision Tree": {
        "clf__max_depth"        : [4, 6, 8, 10, 12, 15, None],
        "clf__min_samples_leaf" : [10, 20, 30, 50],
        "clf__min_samples_split": [10, 20, 40, 60],
        "clf__criterion"        : ["gini", "entropy"],
    },

    # ── Random Forest ────────────────────────────────────────────────────────
    "Random Forest": {
        "clf__n_estimators"     : [100, 200, 300],
        "clf__max_depth"        : [8, 10, 12, 15, None],
        "clf__min_samples_leaf" : [10, 20, 30],
        "clf__min_samples_split": [10, 20, 40],
        "clf__max_features"     : ["sqrt", "log2"],
    },

    # ── XGBoost ──────────────────────────────────────────────────────────────
    "XGBoost": {
        "clf__n_estimators"     : [200, 300, 500],
        "clf__learning_rate"    : [0.01, 0.03, 0.05, 0.1],
        "clf__max_depth"        : [4, 6, 8],
        "clf__subsample"        : [0.7, 0.8, 1.0],
        "clf__colsample_bytree" : [0.7, 0.8, 1.0],
        "clf__reg_alpha"        : [0, 0.1, 0.5],
        "clf__reg_lambda"       : [1.0, 2.0, 5.0],
    },

    # ── LightGBM ─────────────────────────────────────────────────────────────
    "LightGBM": {
        "clf__n_estimators"      : [200, 300, 500],
        "clf__learning_rate"     : [0.01, 0.03, 0.05, 0.1],
        "clf__num_leaves"        : [31, 63, 127],
        "clf__max_depth"         : [6, 8, 10, -1],
        "clf__reg_alpha"         : [0, 0.1, 0.5],
        "clf__reg_lambda"        : [1.0, 2.0, 5.0],
        "clf__min_child_samples" : [20, 30, 50],
    },

    # ── CatBoost ─────────────────────────────────────────────────────────────
    "CatBoost": {
        "clf__iterations"          : [200, 300, 500],
        "clf__learning_rate"       : [0.01, 0.03, 0.05, 0.1],
        "clf__depth"               : [4, 6, 8],
        "clf__l2_leaf_reg"         : [1, 3, 5, 7],
        "clf__bagging_temperature" : [0.0, 0.5, 1.0],
    },

    # ── SVM (CalibratedClassifierCV wrapping LinearSVC) ──────────────────────
    # FIX: Task 3 main wraps LinearSVC inside CalibratedClassifierCV.
    # Param keys must use clf__estimator__ prefix to reach the inner LinearSVC.
    # Document 4 used clf__C which would silently fail to tune anything.
    "SVM": {
        "clf__estimator__C"        : [0.01, 0.1, 1.0, 5.0, 10.0],
        "clf__estimator__max_iter" : [1000, 2000, 3000],
    },
}

# =============================================================================
# ITERATION CONTROL
# =============================================================================

n_iter_map = {
    "Logistic Regression" : 8,
    "Decision Tree"       : 10,
    "Random Forest"       : 10,
    "XGBoost"             : 15,
    "LightGBM"            : 15,
    "CatBoost"            : 15,
    "SVM"                 : 8,
}

# =============================================================================
# RANDOMIZED SEARCH TUNING LOOP
# =============================================================================

print("\n")
print("=" * 110)
print("  HYPERPARAMETER TUNING — TASK 3: OPERATIONAL RISK LEVEL CLASSIFICATION")
print("=" * 110)

tuned_results = {}

for model_name, pipeline in models.items():

    print(f"\n   Tuning → {model_name} ...", end=" ", flush=True)
    t0 = time.time()

    rs = RandomizedSearchCV(
        estimator           = pipeline,
        param_distributions = param_grids[model_name],
        n_iter              = n_iter_map[model_name],
        cv                  = skf,
        scoring             = "f1_macro",
        n_jobs              = N_JOBS,
        random_state        = RANDOM_STATE,
        verbose             = 0,
        refit               = True,
        return_train_score  = False,
    )


    if model_name == "XGBoost":
        rs.fit(X_train, y_train, clf__sample_weight=sample_weights)
    else:
        rs.fit(X_train, y_train)

    train_time    = time.time() - t0
    best_pipeline = rs.best_estimator_

    # Predictions
    y_pred = best_pipeline.predict(X_test)
    y_prob = best_pipeline.predict_proba(X_test)

    # ── Metrics  ─────────────────

    acc        = accuracy_score(y_test, y_pred)

    macro_prec = precision_score(
        y_test, y_pred, average="macro", zero_division=0
    )
    macro_rec  = recall_score(
        y_test, y_pred, average="macro", zero_division=0
    )
    macro_f1   = f1_score(
        y_test, y_pred, average="macro", zero_division=0
    )
    weighted_f1 = f1_score(
        y_test, y_pred, average="weighted", zero_division=0
    )
    per_cls_f1  = f1_score(
        y_test, y_pred, average=None, zero_division=0
    )

    roc_auc = roc_auc_score(
        y_test, y_prob,
        multi_class="ovr",
        average="macro",
        labels=classes
    )

    tuned_results[model_name] = {
        "Accuracy"        : round(acc,            4),
        "Macro Precision" : round(macro_prec,     4),
        "Macro Recall"    : round(macro_rec,      4),
        "Macro F1"        : round(macro_f1,       4),
        "Weighted F1"     : round(weighted_f1,    4),
        "F1 Low"          : round(per_cls_f1[0],  4),
        "F1 Moderate"     : round(per_cls_f1[1],  4),
        "F1 High"         : round(per_cls_f1[2],  4),
        "ROC-AUC"         : round(roc_auc,        4),
        "Best Params"     : rs.best_params_,
        "Train Time (s)"  : round(train_time,     1),
        "_pipeline"       : best_pipeline,
        "_y_pred"         : y_pred,
        "_y_prob"         : y_prob,
    }

    print(
        f"done [{train_time:.1f}s]  "
        f"MacroF1={macro_f1:.4f}  "
        f"ROC-AUC={roc_auc:.4f}"
    )

# =============================================================================
# BEFORE vs AFTER TUNING COMPARISON
# =============================================================================

print("\n")
print("=" * 110)
print("  TUNING COMPARISON — PRE vs POST RandomizedSearchCV (TASK 3)")
print("=" * 110)

rows = []
for m in tuned_results:
    rows.append({
        "Model"               : m,
        "Macro F1 (pre)"      : results[m]["Macro F1"],
        "Macro F1 (tuned)"    : tuned_results[m]["Macro F1"],
        "F1 High (pre)"       : results[m]["F1 High"],
        "F1 High (tuned)"     : tuned_results[m]["F1 High"],
        "AUC (pre)"           : results[m]["ROC-AUC"],
        "AUC (tuned)"         : tuned_results[m]["ROC-AUC"],
        "Train Time (s)"      : tuned_results[m]["Train Time (s)"],
    })

compare_df_tuning = pd.DataFrame(rows).sort_values(
    "Macro F1 (tuned)", ascending=False
).reset_index(drop=True)

print(compare_df_tuning.to_string(index=False))
print("=" * 110)

compare_df_tuning.to_csv(
    "task3_outputs/task3_tuning_comparison.csv", index=False
)
print("\n[✓] Saved → task3_outputs/task3_tuning_comparison.csv")

# =============================================================================
# SAVE BEST HYPERPARAMETERS
# =============================================================================

params_rows = [
    {"Model": m, "Best Params": str(tuned_results[m]["Best Params"])}
    for m in tuned_results
]
pd.DataFrame(params_rows).to_csv(
    "task3_outputs/task3_best_params.csv", index=False
)
print("[✓] Saved → task3_outputs/task3_best_params.csv")

# Print best params — all models
print("\n── Best Hyperparameters — All Models ───────────────────────────────────")
for m in tuned_results:
    print(f"\n   {m}:")
    for param, val in tuned_results[m]["Best Params"].items():
        print(f"     {param:<40} : {val}")

# =============================================================================
# FINAL TUNED MODEL RANKING
# =============================================================================

print("\n")
print("=" * 110)
print("  FINAL TUNED MODEL RANKING — TASK 3: OPERATIONAL RISK CLASSIFICATION")
print("=" * 110)

final_tuned_df = pd.DataFrame([
    {
        "Model"           : m,
        "Accuracy"        : tuned_results[m]["Accuracy"],
        "Macro Precision" : tuned_results[m]["Macro Precision"],
        "Macro Recall"    : tuned_results[m]["Macro Recall"],
        "Macro F1"        : tuned_results[m]["Macro F1"],
        "Weighted F1"     : tuned_results[m]["Weighted F1"],
        "F1 Low"          : tuned_results[m]["F1 Low"],
        "F1 Moderate"     : tuned_results[m]["F1 Moderate"],
        "F1 High"         : tuned_results[m]["F1 High"],
        "ROC-AUC"         : tuned_results[m]["ROC-AUC"],
        "Train Time (s)"  : tuned_results[m]["Train Time (s)"],
    }
    for m in tuned_results
])


final_tuned_df = final_tuned_df.sort_values(
    "Macro F1", ascending=False
).reset_index(drop=True)
final_tuned_df.insert(0, "Rank", final_tuned_df.index + 1)

print(final_tuned_df.to_string(index=False))
print("=" * 110)

best_tuned_name     = final_tuned_df.iloc[0]["Model"]
best_tuned_pipeline = tuned_results[best_tuned_name]["_pipeline"]
best_tuned_y_pred   = tuned_results[best_tuned_name]["_y_pred"]
best_tuned_y_prob   = tuned_results[best_tuned_name]["_y_prob"]

print(f"\n   Best Tuned Model  : {best_tuned_name}")
print(f"   Accuracy          : {tuned_results[best_tuned_name]['Accuracy']:.4f}")
print(f"   Macro Precision   : {tuned_results[best_tuned_name]['Macro Precision']:.4f}")
print(f"   Macro Recall      : {tuned_results[best_tuned_name]['Macro Recall']:.4f}")
print(f"   Macro F1          : {tuned_results[best_tuned_name]['Macro F1']:.4f}")
print(f"   Weighted F1       : {tuned_results[best_tuned_name]['Weighted F1']:.4f}")
print(f"   F1 Low            : {tuned_results[best_tuned_name]['F1 Low']:.4f}")
print(f"   F1 Moderate       : {tuned_results[best_tuned_name]['F1 Moderate']:.4f}")
print(f"   F1 High           : {tuned_results[best_tuned_name]['F1 High']:.4f}")
print(f"   ROC-AUC           : {tuned_results[best_tuned_name]['ROC-AUC']:.4f}")
print(f"   Best Params       : {tuned_results[best_tuned_name]['Best Params']}")
print("=" * 110)

# =============================================================================
# POST-TUNING VISUALISATIONS
# =============================================================================

print("\n── Post-Tuning Visualisations ───────────────────────────────────────────")

PALETTE_TUNED = sns.color_palette("Set2", n_colors=len(tuned_results))
MODEL_ORDER_TUNED = final_tuned_df["Model"].tolist()

# ── Classification report — best tuned model ──────────────────────────────────
print(f"\n── Classification Report — {best_tuned_name} (Tuned) ──────────────────")
print(classification_report(
    y_test, best_tuned_y_pred,
    target_names=CLASS_NAMES,
    digits=4
))

# ── Confusion matrix — best tuned model ───────────────────────────────────────
cm = confusion_matrix(y_test, best_tuned_y_pred)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm, display_labels=CLASS_NAMES
)
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title(
    f"Confusion Matrix — {best_tuned_name} (Tuned)",
    fontsize=11, fontweight="bold"
)
plt.tight_layout()
plt.savefig(
    "task3_outputs/task3_tuned_confusion_matrix.png",
    dpi=150, bbox_inches="tight"
)
plt.close()
print("[✓] Saved → task3_outputs/task3_tuned_confusion_matrix.png")

# ── Per-class F1 bar chart — tuned models ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 5))
per_class_metrics = ["F1 Low", "F1 Moderate", "F1 High"]
per_class_colors  = ["#43A047", "#FFA726", "#EF5350"]
x2     = np.arange(len(MODEL_ORDER_TUNED))
width2 = 0.25

for i, (metric, color) in enumerate(zip(per_class_metrics, per_class_colors)):
    vals = [tuned_results[m][metric] for m in MODEL_ORDER_TUNED]
    bars = ax.bar(x2 + i * width2, vals, width2,
                  label=metric, color=color, alpha=0.85)
    for bar in bars:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.003,
            f"{bar.get_height():.3f}",
            ha="center", va="bottom", fontsize=7, rotation=90
        )
ax.set_xticks(x2 + width2)
ax.set_xticklabels(MODEL_ORDER_TUNED, rotation=25, ha="right", fontsize=9)
ax.set_ylim(0, 1.12)
ax.set_ylabel("F1-Score")
ax.set_title(
    "Task 3 — Per-Class F1 Score (Tuned): Low | Moderate | High Risk",
    fontsize=12, fontweight="bold"
)
ax.legend(loc="upper right", fontsize=9)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(
    "task3_outputs/task3_tuned_per_class_f1.png",
    dpi=150, bbox_inches="tight"
)
plt.close()
print("[✓] Saved → task3_outputs/task3_tuned_per_class_f1.png")

# ── ROC curves — one panel per class  ────────────────
from sklearn.metrics import roc_curve, auc as sklearn_auc
from sklearn.preprocessing import label_binarize

y_test_bin  = label_binarize(y_test, classes=[0, 1, 2])
risk_labels = ["Low Risk", "Moderate Risk", "High Risk"]

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
for class_idx, class_label in enumerate(risk_labels):
    ax = axes[class_idx]
    for i, model_name in enumerate(MODEL_ORDER_TUNED):
        y_prob_model = tuned_results[model_name]["_y_prob"]
        fpr, tpr, _  = roc_curve(y_test_bin[:, class_idx], y_prob_model[:, class_idx])
        roc_val      = sklearn_auc(fpr, tpr)
        ax.plot(fpr, tpr,
                label=f"{model_name} ({roc_val:.3f})",
                color=PALETTE_TUNED[i], lw=1.6)
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    ax.set_xlabel("False Positive Rate")
    ax.set_title(f"OvR — {class_label}", fontsize=10, fontweight="bold")
    ax.legend(fontsize=6.5, loc="lower right")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("True Positive Rate")
fig.suptitle(
    "Task 3 — ROC Curves (Tuned, One-vs-Rest) by Risk Class",
    fontsize=12, fontweight="bold"
)
plt.tight_layout()
plt.savefig(
    "task3_outputs/task3_tuned_roc_curves.png",
    dpi=150, bbox_inches="tight"
)
plt.close()
print("[✓] Saved → task3_outputs/task3_tuned_roc_curves.png")

# ── Heatmap — tuned model metrics ─────────────────────────────────────────────
hm_cols = [
    "Accuracy", "Macro F1", "Weighted F1",
    "F1 Low", "F1 Moderate", "F1 High", "ROC-AUC"
]
hm_df = final_tuned_df.set_index("Model")[hm_cols].astype(float)

fig, ax = plt.subplots(figsize=(13, 5))
sns.heatmap(
    hm_df, annot=True, fmt=".4f", cmap="YlGn",
    linewidths=0.5, ax=ax, annot_kws={"size": 8.5}
)
ax.set_title(
    "Task 3 — Tuned Model Performance Heatmap",
    fontsize=13, fontweight="bold"
)
ax.set_ylabel("")
plt.tight_layout()
plt.savefig(
    "task3_outputs/task3_tuned_heatmap.png",
    dpi=150, bbox_inches="tight"
)
plt.close()
print("[✓] Saved → task3_outputs/task3_tuned_heatmap.png")

# ── CV Macro F1 — post-tuning re-run  ────────────
# Re-running CV on tuned pipelines gives the definitive post-tuning
# generalisation estimate for reporting in the thesis
print("\n── Post-Tuning Cross-Validation Macro F1 (5-Fold) ─────────────────────")

tuned_cv_means = []
tuned_cv_stds  = []

for model_name in MODEL_ORDER_TUNED:
    cv_s = cross_val_score(
        tuned_results[model_name]["_pipeline"],
        X_train, y_train,
        cv=skf, scoring="f1_macro", n_jobs=N_JOBS
    )
    tuned_cv_means.append(cv_s.mean())
    tuned_cv_stds.append(cv_s.std())
    print(f"   {model_name:<22}  CV Macro F1 = {cv_s.mean():.4f} ± {cv_s.std():.4f}")

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(
    MODEL_ORDER_TUNED, tuned_cv_means,
    xerr=tuned_cv_stds,
    color=PALETTE_TUNED[:len(MODEL_ORDER_TUNED)],
    height=0.55, capsize=5, ecolor="gray"
)
ax.set_xlabel("CV Macro F1-Score (mean ± std)")
ax.set_title(
    "Task 3 — Cross-Validated Macro F1: Tuned Models (5-Fold)",
    fontsize=12, fontweight="bold"
)
ax.set_xlim(0, 1.05)
for i, (m, s) in enumerate(zip(tuned_cv_means, tuned_cv_stds)):
    ax.text(m + s + 0.01, i, f"{m:.4f}", va="center", fontsize=8)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(
    "task3_outputs/task3_tuned_cv_macro_f1.png",
    dpi=150, bbox_inches="tight"
)
plt.close()
print("[✓] Saved → task3_outputs/task3_tuned_cv_macro_f1.png")

# =============================================================================
# FINAL SUMMARY — POST TUNING
# =============================================================================

print("\n")
print("=" * 110)
print("  FINAL SUMMARY — TASK 3: POST-TUNING RESULTS")
print("=" * 110)
print(final_tuned_df[[
    "Rank", "Model", "Accuracy",
    "Macro Precision", "Macro Recall",
    "Macro F1", "Weighted F1",
    "F1 Low", "F1 Moderate", "F1 High",
    "ROC-AUC", "Train Time (s)"
]].to_string(index=False))
print("=" * 110)
print(f"\n   Best Tuned Model  : {best_tuned_name}")
print(f"   Macro F1          : {tuned_results[best_tuned_name]['Macro F1']:.4f}")
print(f"   Accuracy          : {tuned_results[best_tuned_name]['Accuracy']:.4f}")
print(f"   ROC-AUC           : {tuned_results[best_tuned_name]['ROC-AUC']:.4f}")
print(f"\n   Saved outputs:")
print("    • task3_outputs/task3_tuning_comparison.csv")
print("    • task3_outputs/task3_best_params.csv")
print("    • task3_outputs/task3_tuned_confusion_matrix.png")
print("    • task3_outputs/task3_tuned_per_class_f1.png")
print("    • task3_outputs/task3_tuned_roc_curves.png")
print("    • task3_outputs/task3_tuned_heatmap.png")
print("    • task3_outputs/task3_tuned_cv_macro_f1.png")
print("=" * 110)
